# Complex Particle Choi-Transfer Spectrum: `N20x40`

This A100 data-generation notebook propagates cumulative complex Choi blocks through the canonical GPU Markov circuit and extracts the particle-transfer Lyapunov spectrum and selected eigenstates of $\mathcal T_p\mathcal T_p^\dagger$ directly from the Hermitian block $A=\Sigma_{LL}$. It runs both the domain-wall slab and homogeneous `DW=False` configurations. No Majorana conversion, pseudoinverse reconstruction, full Choi snapshots, or explicit transfer matrices are stored.

In [ ]:
from __future__ import annotations

from contextlib import contextmanager
import inspect
import json
import sys
import threading
import time
from pathlib import Path

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False
if IN_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive', force_remount=False)

BUNDLE_NAME = 'colab_regularized_choi_transfer_matrix'
def is_bundle_root(path: Path) -> bool:
    return ((path / 'src' / 'classA_U1FGTN_gpu.py').exists()
            and (path / 'src' / 'choi_transfer_observables_gpu.py').exists())

candidates = []
for base in [Path.cwd(), *Path.cwd().parents]:
    candidates.extend([base, base / BUNDLE_NAME])
if IN_COLAB:
    candidates.extend([Path('/content/drive/MyDrive') / BUNDLE_NAME,
                       Path('/content/drive/MyDrive/class_A_fermionic_adaptive_circuit') / BUNDLE_NAME])
BUNDLE_ROOT = next((p.resolve() for p in candidates if is_bundle_root(p.resolve())), None)
if BUNDLE_ROOT is None:
    raise FileNotFoundError(f'Could not locate {BUNDLE_NAME} bundle with Choi helper.')
BUNDLE_SRC_DIR = BUNDLE_ROOT / 'src'
sys.path.insert(0, str(BUNDLE_SRC_DIR))

import numpy as np
import torch
from tqdm.auto import tqdm
from classA_U1FGTN_gpu import classA_U1FGTN_gpu
from choi_transfer_observables_gpu import (
    CANONICAL_ENTRY_POINT, CHOI_FORMULA_VERSION, HELPER_VERSION,
    HOLE_SPECTRUM_RELATION, OBSERVABLE, PARTICLE_TRANSFER_DEFINITION,
    ParticleChoiTransferObserver, particle_spectrum_from_choi_blocks,
    save_npz_atomic, write_json_atomic,
)
HEARTBEAT_INTERVAL_S = 60.0


@contextmanager
def progress_heartbeat(progress_bar, config_label: str, *, interval_s: float = HEARTBEAT_INTERVAL_S):
    if interval_s <= 0:
        raise ValueError('heartbeat interval must be positive')
    stop_event = threading.Event()
    started = time.perf_counter()

    def emit() -> None:
        while not stop_event.wait(interval_s):
            elapsed_s = time.perf_counter() - started
            progress_bar.set_postfix_str(f'{config_label} elapsed={elapsed_s:.0f}s', refresh=True)
            progress_bar.write(f'[heartbeat] {config_label} still running; elapsed={elapsed_s:.0f}s')

    worker = threading.Thread(target=emit, name='colab-progress-heartbeat', daemon=True)
    worker.start()
    try:
        yield
    finally:
        stop_event.set()
        worker.join()


EXPECTED_OBSERVABLE = 'complex_particle_choi_transfer_spectrum'
if OBSERVABLE != EXPECTED_OBSERVABLE:
    raise RuntimeError(f'Unexpected Choi-transfer observable label: {OBSERVABLE}')
run_signature = inspect.signature(classA_U1FGTN_gpu.run_markov_circuit)
for name in ('track_choi', 'choi_observer', 'choi_observer_cycles', 'choi_singular_tol', 'meas_slab_only'):
    if name not in run_signature.parameters:
        raise RuntimeError(f'Stale GPU class missing {name}: {run_signature}')
if not torch.cuda.is_available():
    raise RuntimeError('CUDA is required for this campaign.')
torch.set_grad_enabled(False)
print(f'[setup] bundle root={BUNDLE_ROOT}')
print(f'[setup] helper={HELPER_VERSION}, gpu={torch.cuda.get_device_name(0)}')
if torch.cuda.get_device_properties(0).total_memory / 1024**3 < 35.0:
    print('[warn] Campaign is sized for an A100 40 GB runtime.')


In [ ]:
NX, NY = 20, 40
CYCLES = 40
SAMPLES = 10
NSHELL = 1
ALPHA_1 = 1.0
DTYPE, DEVICE, BACKEND = 'complex128', 'cuda:0', 'local'
SEQUENCE = 'raster_y'
PROTOCOL = 'perfect_correction'
OBSERVER_CYCLES = [1, 5, 10, 20, 40]
ENDPOINT_TOL = 1e-12
CHOI_SPECTRAL_TOL = 1e-10
CHOI_SINGULAR_TOL = 1e-10
BATCH_SIZE = 1
PRODUCTION_CONFIGS = [
    {'run_label': 'DW1_dwtrunc1_slab_a1-1_a2-30', 'DW': True, 'dw_truncation': True,
     'meas_slab_only': True, 'alpha_1': ALPHA_1, 'alpha_2': 30.0},
    {'run_label': 'DW0_homogeneous_a1-1', 'DW': False, 'dw_truncation': False,
     'meas_slab_only': True, 'alpha_1': ALPHA_1, 'alpha_2': 1.0},
]
OUT_ROOT = BUNDLE_ROOT / 'gpu_data' / 'complex_particle_choi_transfer'
OUT_ROOT.mkdir(parents=True, exist_ok=True)


## Mandatory CUDA Smoke Test

The preflight validates the regulated rank-one Choi update, recovers finite particle singular exponents from synthetic pure Choi tensors, checks exact particle zero/pole sectors, exercises all three protocol paths in slab-only mode, and verifies the homogeneous `DW=False` full-basis path.

In [ ]:
def full_sigma(state):
    return torch.cat((torch.cat((state['LL'], state['LR']), dim=-1),
                      torch.cat((state['LR'].mH, state['RR']), dim=-1)), dim=-2)

def formula_checks() -> None:
    torch.manual_seed(21)
    n = 7
    cycle = 4
    tp = 0.25 * torch.randn((n, n), dtype=torch.complex128, device=DEVICE)
    eye = torch.eye(n, dtype=tp.dtype, device=tp.device)
    hh = tp @ tp.mH
    a = torch.linalg.solve(eye + hh, eye - hh)
    measured = particle_spectrum_from_choi_blocks(a.unsqueeze(0), cycle=cycle,
        endpoint_tol=ENDPOINT_TOL, choi_spectral_tol=CHOI_SPECTRAL_TOL, n_eigenstates=3)
    expected = torch.sort(torch.log(torch.linalg.svdvals(tp)) / float(cycle)).values
    assert torch.allclose(measured['spectrum'][0], expected, atol=1e-11, rtol=1e-10)
    tt = tp @ tp.mH
    for k in range(3):
        vec = measured['eigenstates'][0, :, k]
        lam = measured['eigenstate_exponents'][0, k]
        sval2 = torch.exp(2.0 * float(cycle) * lam)
        assert torch.linalg.vector_norm(tt @ vec - sval2 * vec) < 1e-10
    endpoint_a = torch.diag(torch.tensor([1.0, -1.0, 0.0, 0.2, -0.2], device=DEVICE, dtype=torch.float64)).to(torch.complex128)
    endpoint = particle_spectrum_from_choi_blocks(endpoint_a.unsqueeze(0), cycle=1,
        endpoint_tol=ENDPOINT_TOL, choi_spectral_tol=CHOI_SPECTRAL_TOL, n_eigenstates=3)
    assert int(endpoint['particle_zero_count'][0]) == 1
    assert int(endpoint['particle_pole_count'][0]) == 1
    assert int(endpoint['hole_zero_count'][0]) == 1 and int(endpoint['hole_pole_count'][0]) == 1
    assert torch.isneginf(endpoint['spectrum']).any() and torch.isposinf(endpoint['spectrum']).any()
    print('[smoke] complex particle spectrum, T_p T_p^dagger eigenstates, and endpoint checks passed')

def kernel_checks() -> None:
    model = classA_U1FGTN_gpu(2, 2, DW=False, nshell=1, device=DEVICE, dtype=DTYPE, backend='local')
    n = model.Nlayer
    torch.manual_seed(19)
    eye = torch.eye(n, dtype=model.dtype, device=model.device)
    for eta1 in (-1.0, 1.0):
        for eta2 in (-1.0, 1.0):
            chi = torch.randn(n, dtype=model.dtype, device=model.device)
            chi = chi / torch.linalg.vector_norm(chi)
            p = chi[:, None] * chi.conj()[None, :]
            q = eye - p
            x = torch.randn(n, n, dtype=model.dtype, device=model.device)
            ll = 0.5 * (x + x.mH)
            lr = torch.randn(n, n, dtype=model.dtype, device=model.device)
            x = torch.randn(n, n, dtype=model.dtype, device=model.device)
            rr = 0.5 * (x + x.mH)
            d = torch.vdot(chi, rr @ chi) - eta1
            if torch.abs(d) < 0.1:
                rr = rr + 0.5 * p
                d = torch.vdot(chi, rr @ chi) - eta1
            state = {'LL': ll[None].clone(), 'LR': lr[None].clone(), 'RR': rr[None].clone(),
                     'singular_tol': 1e-12, 'min_abs_d': float('inf'), 'min_abs_d_context': None}
            model._choi_apply_rank_one(state, torch.tensor([0], device=model.device), chi,
                                       torch.tensor([eta1], device=model.device),
                                       torch.tensor([eta2], device=model.device))
            ref = (ll - lr @ p @ lr.mH / d,
                   lr @ q - lr @ p @ rr @ q / d,
                   eta2 * p + q @ rr @ q - q @ rr @ p @ rr @ q / d)
            err = max(float(torch.linalg.matrix_norm(state[k][0] - target).item())
                      for k, target in zip(('LL', 'LR', 'RR'), ref))
            assert err < 1e-10, (eta1, eta2, err)
            pure = model._init_choi_state(1, singular_tol=1e-12)
            model._choi_apply_rank_one(pure, torch.tensor([0], device=model.device), chi,
                                       torch.tensor([eta1], device=model.device),
                                       torch.tensor([eta2], device=model.device))
            assert torch.linalg.matrix_norm(pure['LL'][0] - eta1 * p) < 1e-10
            assert torch.linalg.matrix_norm(pure['LR'][0] - q) < 1e-10
            assert torch.linalg.matrix_norm(pure['RR'][0] - eta2 * p) < 1e-10
            sigma = full_sigma(pure)
            assert torch.linalg.matrix_norm(sigma - sigma.mH) < 1e-10
            invol = torch.linalg.matrix_norm(sigma @ sigma - torch.eye(2*n, dtype=model.dtype, device=model.device))
            assert float(invol.item()) < 1e-10, (eta1, eta2, float(invol.item()))
    print('[smoke] rank-one formula, identity composition, Hermiticity, and involution checks passed')

def run_protocol_smoke(protocol: str, *, dw: bool) -> dict:
    samples = 1 if protocol == 'post_selection' else 2
    model = classA_U1FGTN_gpu(4, 8, DW=dw, nshell=1, alpha_1=1.0, alpha_2=30.0 if dw else 1.0,
                              dw_truncation=dw, device=DEVICE, dtype=DTYPE, backend=BACKEND)
    active_indices = model.active_top_layer_indices(meas_slab_only=True)
    n_active = int(active_indices.numel())
    active_np = active_indices.detach().cpu().numpy()
    physical_snapshots = {}
    def physical_observer(*, cycle, G, **_):
        if cycle in (0, 2):
            physical_snapshots[cycle] = G.detach().cpu().numpy().copy()
    observer = ParticleChoiTransferObserver(samples_expected=samples, cycles=[1, 2], nlayer=n_active,
        endpoint_tol=ENDPOINT_TOL, choi_spectral_tol=CHOI_SPECTRAL_TOL, validate_choi=True,
        retain_sigma_ll=True, active_top_layer_indices=active_np, full_nlayer=model.Nlayer)
    kwargs = {'postselect': protocol == 'post_selection',
              'perfect_correction': protocol == 'perfect_correction'}
    result = model.run_markov_circuit(G_history=False, progress=False, cycles=2, samples=samples,
        save=False, return_data=False, n_a=0.5, sequence=SEQUENCE, meas_slab_only=True, batch_size=samples,
        cycle_observer=physical_observer, track_choi=True, choi_observer=observer, choi_observer_cycles=[1, 2],
        choi_singular_tol=CHOI_SINGULAR_TOL, **kwargs)
    observer.assert_complete()
    assert bool(result['meas_slab_only_effective']) == bool(dw)
    assert observer.spectra.shape == (samples, 2, n_active)
    assert np.array_equal(observer.active_top_layer_indices, active_np)
    assert not np.isnan(observer.spectra).any()
    assert np.isfinite(observer.eigenstate_exponents).all()
    if dw:
        exterior_np = np.setdiff1d(np.arange(model.Nlayer), active_np)
        assert len(model._sequence_helper(SEQUENCE, meas_slab_only=True)['coords_for_len']) == n_active // 2
        for G in physical_snapshots.values():
            assert np.max(np.abs(G[:, exterior_np][:, :, active_np])) < 1e-9
            ext = G[:, exterior_np][:, :, exterior_np]
            assert np.max(np.abs(ext - np.eye(len(exterior_np))[None] * np.diagonal(ext, axis1=1, axis2=2)[:, None, :])) < 1e-9
            assert np.max(np.abs(np.abs(np.diagonal(ext, axis1=1, axis2=2)) - 1.0)) < 1e-9
        assert np.max(np.abs(physical_snapshots[2][:, exterior_np][:, :, exterior_np] - physical_snapshots[0][:, exterior_np][:, :, exterior_np])) < 1e-9
        if protocol == 'post_selection':
            assert np.max(np.abs(np.diagonal(physical_snapshots[0][:, exterior_np][:, :, exterior_np], axis1=1, axis2=2) + 1.0)) < 1e-9
    else:
        assert n_active == model.Nlayer
    assert np.nanmax(observer.hermiticity_residual) < 1e-10
    assert np.nanmax(observer.involution_residual) < 1e-8
    assert min(item['min_abs_d'] for item in result['choi_diagnostics']) > CHOI_SINGULAR_TOL
    sensitivity = observer.endpoint_sensitivity_report([1e-10, 1e-12, 1e-14])
    baseline = sensitivity['1.0e-12']
    for key, rows in sensitivity.items():
        for cycle, row in rows.items():
            assert row['eigenstate_indices'] == baseline[cycle]['eigenstate_indices'], (dw, protocol, key, cycle, 'indices')
            delta = np.max(np.abs(np.asarray(row['eigenstate_exponents']) - np.asarray(baseline[cycle]['eigenstate_exponents'])))
            assert delta < 1e-8, (dw, protocol, key, cycle, delta)
    print(f'[smoke] DW={int(dw)} {protocol}: endpoint counts zero={observer.particle_zero_count.tolist()} pole={observer.particle_pole_count.tolist()}')
    return sensitivity

formula_checks()
kernel_checks()
smoke_sensitivity = {
    f'DW1_{protocol}': run_protocol_smoke(protocol, dw=True)
    for protocol in ('post_selection', 'perfect_correction', 'sampled_feedback')
}
smoke_sensitivity['DW0_perfect_correction'] = run_protocol_smoke('perfect_correction', dw=False)
print('[smoke] endpoint sensitivity report')
print(json.dumps(smoke_sensitivity, indent=2))
torch.cuda.empty_cache()


## Production Extraction

The same particle-spectrum observable is evaluated on the reduced domain-wall slab and on the full homogeneous topological system.

In [ ]:
campaign_runs = []
campaign_bar = tqdm(PRODUCTION_CONFIGS, total=len(PRODUCTION_CONFIGS), desc='Choi-spectrum configurations', unit='config', leave=True, dynamic_ncols=True)
for cfg in campaign_bar:
    label = cfg['run_label']
    campaign_bar.set_postfix_str(f'{label} elapsed=0s', refresh=True)
    model = classA_U1FGTN_gpu(NX, NY, DW=cfg['DW'], nshell=NSHELL,
        alpha_1=cfg['alpha_1'], alpha_2=cfg['alpha_2'], dw_truncation=cfg['dw_truncation'],
        device=DEVICE, dtype=DTYPE, backend=BACKEND)
    active_indices = model.active_top_layer_indices(meas_slab_only=cfg['meas_slab_only'])
    active_np = active_indices.detach().cpu().numpy()
    n_active = int(active_indices.numel())
    observer = ParticleChoiTransferObserver(samples_expected=SAMPLES, cycles=OBSERVER_CYCLES, nlayer=n_active,
        endpoint_tol=ENDPOINT_TOL, choi_spectral_tol=CHOI_SPECTRAL_TOL, validate_choi=False,
        active_top_layer_indices=active_np, full_nlayer=model.Nlayer)
    started = time.perf_counter()
    with progress_heartbeat(campaign_bar, label):
        result = model.run_markov_circuit(G_history=False, progress=False, cycles=CYCLES,
            postselect=False, perfect_correction=True, samples=SAMPLES, save=False, return_data=False,
            n_a=0.5, sequence=SEQUENCE, meas_slab_only=cfg['meas_slab_only'], batch_size=BATCH_SIZE, track_choi=True,
            choi_observer=observer, choi_observer_cycles=OBSERVER_CYCLES, choi_singular_tol=CHOI_SINGULAR_TOL)
    elapsed_s = float(time.perf_counter() - started)
    observer.assert_complete()
    expected_effective = bool(cfg['DW'] and cfg['dw_truncation'] and cfg['meas_slab_only'])
    assert bool(result['meas_slab_only_effective']) == expected_effective
    assert observer.spectra.shape == (SAMPLES, len(OBSERVER_CYCLES), n_active)
    assert observer.eigenstates_near_zero.shape == (SAMPLES, len(OBSERVER_CYCLES), n_active, 3)
    assert observer.eigenstate_exponents.shape == (SAMPLES, len(OBSERVER_CYCLES), 3)
    assert observer.finite_eigenstate_count.shape == (SAMPLES, len(OBSERVER_CYCLES))
    out_dir = OUT_ROOT / cfg['run_label']
    out_dir.mkdir(parents=True, exist_ok=True)
    metadata = {
        'canonical_dynamics_entry_point': CANONICAL_ENTRY_POINT,
        'observable': OBSERVABLE,
        'helper_version': HELPER_VERSION,
        'choi_initialization': 'Sigma_LL=0, Sigma_LR=I, Sigma_RR=0',
        'choi_formula_version': CHOI_FORMULA_VERSION,
        'transfer_basis': 'reduced_topological_slab_complex_fermion_basis' if expected_effective else 'full_top_layer_complex_fermion_basis',
        'transfer_object': 'T_p',
        'particle_transfer_definition': PARTICLE_TRANSFER_DEFINITION,
        'spectrum_extraction': 'Cayley_singular_values_from_Hermitian_Sigma_LL',
        'hole_spectrum_reconstruction': HOLE_SPECTRUM_RELATION,
        'endpoint_policy': 'retain_exact_positive_and_negative_infinite_particle_exponents',
        'eigenstate_operator': 'T_p T_p^dagger',
        'near_zero_eigenstate_policy': 'three_finite_near_zero_eigenstates_of_T_p_T_p_dagger',
        'endpoint_tol': ENDPOINT_TOL,
        'choi_spectral_tol': CHOI_SPECTRAL_TOL,
        'choi_singular_tol': CHOI_SINGULAR_TOL,
        'observation_cycles': OBSERVER_CYCLES,
        'protocol': PROTOCOL,
        'configuration': {'Nx': NX, 'Ny': NY, 'cycles': CYCLES, 'samples': SAMPLES, 'nshell': NSHELL,
            'DW': cfg['DW'], 'dw_truncation': cfg['dw_truncation'], 'meas_slab_only': cfg['meas_slab_only'],
            'alpha_1': cfg['alpha_1'], 'alpha_2': cfg['alpha_2'], 'dtype': DTYPE,
            'device': DEVICE, 'backend': BACKEND, 'sequence': SEQUENCE, 'batch_size': BATCH_SIZE},
        'elapsed_s': elapsed_s,
        'meas_slab_only_requested': bool(cfg['meas_slab_only']),
        'meas_slab_only_effective': bool(result['meas_slab_only_effective']),
        'exterior_preparation': 'born_sampled_onsite_before_cycle_0' if expected_effective else None,
        'postselected_exterior_preparation': 'deterministic_vacuum_onsite_before_cycle_0' if expected_effective else None,
        'Nlayer_active': n_active,
        'active_top_layer_indices': active_np.tolist(),
        'choi_diagnostics': result['choi_diagnostics'],
        'smoke_endpoint_sensitivity': smoke_sensitivity,
        'full_choi_or_transfer_matrices_saved': False,
    }
    metadata_json = np.asarray(json.dumps(metadata, sort_keys=True))
    save_npz_atomic(out_dir / 'particle_choi_transfer_spectra.npz', **observer.spectra_payload(), metadata_json=metadata_json)
    save_npz_atomic(out_dir / 'particle_choi_transfer_eigenstates_near_zero.npz', **observer.eigenstates_payload(), metadata_json=metadata_json)
    save_npz_atomic(out_dir / 'particle_choi_transfer_diagnostics.npz', **observer.diagnostics_payload(), metadata_json=metadata_json)
    write_json_atomic(out_dir / 'run_summary.json', metadata)
    campaign_runs.append({'run_label': cfg['run_label'], 'output_directory': str(out_dir), 'metadata': metadata,
        'files': ['particle_choi_transfer_spectra.npz', 'particle_choi_transfer_eigenstates_near_zero.npz',
                  'particle_choi_transfer_diagnostics.npz', 'run_summary.json']})
    print(f"[production] {cfg['run_label']} completed in {elapsed_s:.1f}s; basis dimension={n_active}")
    del model, observer
    torch.cuda.empty_cache()

manifest = {'status': 'completed', 'observable': OBSERVABLE, 'output_root': str(OUT_ROOT),
    'production_runs': campaign_runs, 'canonical_dynamics_entry_point': CANONICAL_ENTRY_POINT}
write_json_atomic(OUT_ROOT / 'campaign_manifest.json', manifest)
print(json.dumps(manifest, indent=2))


In [ ]:
from IPython.display import Javascript, display
try:
    display(Javascript('google.colab.kernel.disconnect()'))
except Exception as exc:
    print(f'[disconnect] skipped: {exc}')
